# GEMS E6 V2 - GeoStructureNet from raw data only

One architecture, raw inputs only, one Drive namespace: `GEMS_2/e6_v2_from_raw/`.

Raw GeoDAWN raster + fault labels + official template + USGS 1 m DEM tiles -> terrain / FFT potential-field / DEM features -> datasets and splits -> dual-encoder reference (from scratch, frozen) -> E6 masked three-way model -> pooled DTI -> 3-fold full-region inference -> `submission_e6_final.tif` -> **manual** upload.

No earlier cache, checkpoint, prediction or submission is read. Every stage is resumable: re-run the notebook from the top after a disconnect; finished artifacts with a matching fingerprint are reused. Docs: `docs/e6_v2/` (start with `COLAB_EXECUTION.md`).

## 00 CONFIG
Everything else is in `configs/e6_v2/e6_v2.yaml`. Overrides here are applied on top of it (nested dict) and are part of every fingerprint, so set them before the first run.

In [ ]:
REPO = "mybenkhadda/GEMS---drivendata"
REF = "main"                      # branch or commit SHA
CODE = "/content/GEMS_E6V2"
DRIVE_ROOT = "/content/drive/MyDrive/GEMS_2"
CONFIG_FILE = "configs/e6_v2/e6_v2.yaml"
OVERRIDES = {
    "paths": {"drive_root": DRIVE_ROOT},
    # "dem": {"budget_minutes": 300},            # stop the DEM stage cleanly after 5 h; re-run to continue
    # "dem": {"vertical_unit_override": "metre"}, # ONLY after reading the USGS metadata yourself
}

## 01 MOUNT DRIVE / LOAD SOURCE CODE

In [ ]:
import subprocess, sys
from pathlib import Path
from google.colab import drive
from IPython.display import display

drive.mount("/content/drive")
assert Path(DRIVE_ROOT).is_dir(), f"{DRIVE_ROOT} not found"
if not (Path(CODE) / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, f"https://github.com/{REPO}.git", CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "fetch", "--depth", "1", "origin", REF], check=True)
    subprocess.run(["git", "-C", CODE, "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE + "[colab]", "rasterio", "scikit-image"], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
HEAD = subprocess.check_output(["git", "-C", CODE, "rev-parse", "HEAD"], text=True).strip()

from gems_bench.e6_v2 import E6V2Pipeline, load_config
from gems_bench.e6_v2.spatial_splits import DISC_UNITS, GEO_UNITS
CFG = load_config(Path(CODE) / CONFIG_FILE, OVERRIDES)
PIPE = E6V2Pipeline(CFG)
print("code commit:", HEAD)
print("namespace:", PIPE.paths.root)

## 02 RAW DATA INVENTORY
Locates the GeoDAWN raster, fault labels, template and every DEM tile (with its FGDC metadata XML), reads headers and checksums them. Writes `manifests/raw_data_inventory.{json,csv}`. The first run hashes the large GeoDAWN files once; unchanged files are not re-hashed.

In [ ]:
INV = PIPE.inventory()
print(INV)
import pandas as pd
inv_df = pd.read_csv(PIPE.paths.manifests / "raw_data_inventory.csv")
display(inv_df.groupby(["type", "validation_status"]).size().rename("n").reset_index())
display(inv_df[inv_df.type.str.startswith("geodawn")])

## 03 PREFLIGHT AND INPUT VALIDATION
Reads the actual headers (bands, CRS, resolution, shape, template grid, DEM resolution / CRS / vertical unit / overlap with the grid), packages, GPU and disk. Stops with ONE report listing every blocker (`reports/preflight_report.md`). Nothing expensive runs before this passes.

In [ ]:
REPORT = PIPE.preflight(raise_on_block=False)
print("STATUS:", REPORT["status"])
for w in REPORT["warnings"]:
    print("warning:", w)
if REPORT["blockers"]:
    raise RuntimeError("E6 V2 preflight BLOCKED:\n" + "\n".join(" - " + b for b in REPORT["blockers"]))
display(PIPE.status())

## 04 RAW GEOPHYSICS / FFT FEATURES
Base masks (footprint, labels, template), 100 m terrain features (branch 1) and magnetic / gravity FFT features with 2 pi wavenumber scaling, padding, taper and nodata handling plus the raw subsurface bands (branch 2). Robust quantile normalisation is saved next to each cache.

In [ ]:
GEO = PIPE.prepare_geophysics()
print(GEO)
display(pd.read_csv(PIPE.paths.terrain / "feature_registry.csv"))
display(pd.read_csv(PIPE.paths.potential / "feature_registry.csv"))

## 05 DEM MOSAIC / 2m PROCESSING
Per 5 km window (+ 420 m halo): deterministic mosaic of the 1 m tiles warped and averaged to 2 m in EPSG:32611, vertical units converted to metres, seams between DEM projects masked, 2 m operators, 100 m statistics. Each window is saved atomically; a disconnect loses only the windows in flight. With `dem.budget_minutes > 0` the cell stops cleanly - run it again to continue. This is the longest CPU stage; its runtime has not been measured.

In [ ]:
DEM = PIPE.prepare_dem(assemble=False)
print(DEM)
if DEM["status"] == "incomplete":
    raise RuntimeError(f"DEM windows left: {DEM['windows_left']} - re-run this cell (finished windows are kept)")

## 06 DEM 100m SUMMARIES
Assembles the 22 DEM statistics and the separate measured `dem_coverage` in [0, 1] on the 100 m grid and saves their normalisation.

In [ ]:
DEM100 = PIPE.prepare_dem(assemble=True)
print(DEM100)
PIPE.figures()
from IPython.display import Image
display(Image(str(PIPE.paths.figures / "dem_coverage.png")))

## 07 NORMALIZATION / DATASET CREATION
Normalised float16 input stacks per branch on the padded grid, the pooled context stack and the masks (`datasets/`). Cells with any NaN DEM statistic get coverage 0 (exact fallback to the reference).

In [ ]:
DS = PIPE.prepare_datasets(include_splits=False)
print(DS)

## 08 GEOGRAPHIC AND DISCOVERY SPLITS
Three geographic folds (blocks, k-means clusters, buffers, inner validation) and three withheld fault-system discovery splits, generated from the raw labels.

In [ ]:
SPLITS = PIPE.prepare_splits()
print(SPLITS)
display(pd.read_csv(PIPE.paths.splits / "split_summary.csv"))

## 09 MODEL INITIALIZATION / BASELINE TRAINING
Stages A + B for the three geographic folds: the dual-encoder reference (terrain + field) trained from scratch, then written once as the immutable `checkpoints/<unit>/reference.pt`. GPU required. Resumes from `A_last.pt` after a disconnect.

In [ ]:
REF_SHA = PIPE.train_reference(GEO_UNITS)
print(REF_SHA)

## 10 E6 MASKED THREE-WAY TRAINING
Stages C (initialise from the reference + parity check), D (DEM encoder and DEM gate paths only) and E (controlled joint fine-tune with 30 % DEM modality dropout) for the geographic folds. Uncovered pixels are the frozen reference exactly (checked after E).

In [ ]:
E6_GEO = PIPE.train_e6(GEO_UNITS)
print(E6_GEO)

## 11 DISCOVERY TRAINING
Stages A - E for the three withheld fault-system splits (their own references, from scratch).

In [ ]:
E6_DISC = PIPE.train(DISC_UNITS)
print(E6_DISC)

## 12 DTI EVALUATION
Canonical DTI (300 m, alpha 0.2, beta 0.8), components pooled across folds / splits, for the probabilities and for the fixed area-budget rule; DEM coverage slices; short-fault recall; gate weights. Tables in `metrics/`, figures in `figures/`.

In [ ]:
EVAL = PIPE.evaluate()
display(EVAL["pooled"])
display(pd.read_csv(PIPE.paths.metrics / "coverage_slices.csv"))
display(pd.read_csv(PIPE.paths.metrics / "short_fault_recall.csv").query("unit == 'pooled'"))
for f in ("loss_curves", "gate_weights"):
    display(Image(str(PIPE.paths.figures / f"{f}.png")))

## 13 FULL-REGION INFERENCE
Each geographic-fold E6 model predicts the whole grid (`predictions/e6_fold{0,1,2}_full.tif`, reused while the checkpoint is unchanged); the fixed combination is their mean (`predictions/e6_full_probability.tif`).

In [ ]:
FULL = PIPE.infer_full_region()
print(FULL)
display(Image(str(PIPE.paths.figures / "prediction_map.png")))

## 14 AREA-BUDGET EXPORT
Writes `submissions/e6_continuous.tif`, `submissions/e6_area_budget_binary.tif` and `submissions/submission_e6_final.tif` (= the `export.final` policy of the config), each validated against the template and read back.

In [ ]:
SUB = PIPE.export_submission()
print(SUB)

## 15 OFFICIAL SUBMISSION VALIDATION
Re-opens the final file and checks it against the official template again (band count, float32, CRS, transform, shape, NaN pattern, range, binary values for the binary policy, sha256 equal to the export record). **Before uploading, read the competition's submission-format page yourself** and confirm the policy in `export.final` (continuous probabilities vs binary map) is accepted.

In [ ]:
VAL = PIPE.validate_submission()
display(VAL["checks"])
print("policy:", CFG["export"]["final"], "| ok:", VAL["ok"], "| sha256:", VAL["sha256"])
if not VAL["ok"]:
    raise RuntimeError("final submission failed validation - do not upload")

## 16 DOWNLOAD SUBMISSION
Downloads the validated file to your computer. Nothing is uploaded to DrivenData automatically.

In [ ]:
from gems_bench.e6_v2.cache import sha256_file
FINAL = PIPE.paths.submissions / "submission_e6_final.tif"
assert VAL["ok"] and sha256_file(FINAL) == VAL["sha256"], "file changed since validation"
print("upload manually:", FINAL, "sha256", VAL["sha256"])
from google.colab import files
files.download(str(FINAL))